In [2]:
############## PLEASE RUN THIS CELL FIRST! ###################

# import everything and define a test runner function
from importlib import reload
from helper import run
import ecc
import helper
import tx
import script

In [3]:
from tx import Tx
from io import BytesIO
raw_tx = ('0100000001813f79011acb80925dfe69b3def355fe914bd1d96a3f5f71bf8303c6a989c7d1000000006b483045022100ed81ff192e75a3fd2304004dcadb746fa5e24c5031ccfcf21320b0277457c98f02207a986d955c6e0cb35d446a89d3f56100f4d7f67801c31967743a9c8e10615bed01210349fc4e631e3624a545de3f89f5d8684c7b8138bd94bdd531d2e213bf016b278afeffffff02a135ef01000000001976a914bc3b654dca7e56b04dca18f2566cdaf02e8d9ada88ac99c39800000000001976a9141c4bc762dd5423e332166702cb75f40df79fea1288ac19430600')
stream = BytesIO(bytes.fromhex(raw_tx))
transaction = Tx.parse(stream)
print(transaction.fee() >= 0)

True


In [4]:
from ecc import S256Point, Signature
sec = bytes.fromhex('0349fc4e631e3624a545de3f89f5d8684c7b8138bd94bdd531d2e213bf016b278a')
der = bytes.fromhex('3045022100ed81ff192e75a3fd2304004dcadb746fa5e24c5031ccfcf21320b0277457c98f02207a986d955c6e0cb35d446a89d3f56100f4d7f67801c31967743a9c8e10615bed')
z = 0x27e0c5994dec7824e56dec6b2fcb342eb7cdb0d0957c2fce9882f715e85d81a6
point = S256Point.parse(sec)
signature = Signature.parse(der)
print(point.verify(z, signature))

True


In [5]:
from helper import hash256
modified_tx = bytes.fromhex('0100000001813f79011acb80925dfe69b3def355fe914bd1d96a3f5f71bf8303c6a989c7d1000000001976a914a802fc56c704ce87c42d7c92eb75e7896bdc41ae88acfeffffff02a135ef01000000001976a914bc3b654dca7e56b04dca18f2566cdaf02e8d9ada88ac99c39800000000001976a9141c4bc762dd5423e332166702cb75f40df79fea1288ac1943060001000000')
h256 = hash256(modified_tx)
z = int.from_bytes(h256, 'big')
print(hex(z))

0x27e0c5994dec7824e56dec6b2fcb342eb7cdb0d0957c2fce9882f715e85d81a6


In [6]:
from ecc import S256Point, Signature
sec = bytes.fromhex('0349fc4e631e3624a545de3f89f5d8684c7b8138bd94bdd531d2e213bf016b278a')
der = bytes.fromhex('3045022100ed81ff192e75a3fd2304004dcadb746fa5e24c5031ccfcf21320b0277457c98f02207a986d955c6e0cb35d446a89d3f56100f4d7f67801c31967743a9c8e10615bed')
z = 0x27e0c5994dec7824e56dec6b2fcb342eb7cdb0d0957c2fce9882f715e85d81a6
point = S256Point.parse(sec)
signature = Signature.parse(der)
point.verify(z, signature)

True

### Exercise 1

Write the `sig_hash` method for the `Tx` class.

#### Make [this test](/edit/code-ch07/tx.py) pass: `tx.py:TxTest:test_sig_hash`

In [7]:
# Exercise 1

reload(tx)
run(tx.TxTest("test_sig_hash"))

.
----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


### Exercise 2

Write the `verify_input` method for the `Tx` class. You will want to use the `TxIn.script_pubkey()`, `Tx.sig_hash()` and `Script.evaluate()` methods.

#### Make [this test](/edit/code-ch07/tx.py) pass: `tx.py:TxTest:test_verify_p2pkh`

In [8]:
# Exercise 2

reload(tx)

def verify_input(self, input_index):
    tx_in = self.tx_ins[input_index]
    script_pubkey = tx_in.script_pubkey(testnet=self.testnet)
    z = self.sig_hash(input_index)
    combined = tx_in.script_sig + script_pubkey
    return combined.evaluate(z)

tx.Tx.verify_input = verify_input

run(tx.TxTest("test_verify_p2pkh"))

.
----------------------------------------------------------------------
Ran 1 test in 0.160s

OK


In [9]:
from helper import decode_base58, SIGHASH_ALL
from script import p2pkh_script, Script
from tx import TxIn, TxOut, Tx
prev_tx = bytes.fromhex('0d6fe5213c0b3291f208cba8bfb59b7476dffacc4e5cb66f6eb20a080843a299')
prev_index = 13
tx_in = TxIn(prev_tx, prev_index)
tx_outs = []
change_amount = int(0.33*100000000)
change_h160 = decode_base58('mzx5YhAH9kNHtcN481u6WkjeHjYtVeKVh2')
change_script = p2pkh_script(change_h160)
change_output = TxOut(amount=change_amount, script_pubkey=change_script)
target_amount = int(0.1*100000000)
target_h160 = decode_base58('mnrVtF8DWjMu839VW3rBfgYaAfKk8983Xf')
target_script = p2pkh_script(target_h160)
target_output = TxOut(amount=target_amount, script_pubkey=target_script)
tx_obj = Tx(1, [tx_in], [change_output, target_output], 0, True)
print(tx_obj)

tx: cd30a8da777d28ef0e61efe68a9f7c559c1d3e5bcd7b265c850ccb4068598d11
version: 1
tx_ins:
0d6fe5213c0b3291f208cba8bfb59b7476dffacc4e5cb66f6eb20a080843a299:13
tx_outs:
33000000:OP_DUP OP_HASH160 d52ad7ca9b3d096a38e752c2018e6fbc40cdf26f OP_EQUALVERIFY OP_CHECKSIG
10000000:OP_DUP OP_HASH160 507b27411ccf7f16f10297de6cef3f291623eddf OP_EQUALVERIFY OP_CHECKSIG
locktime: 0


In [10]:
from ecc import PrivateKey
from helper import SIGHASH_ALL
z = transaction.sig_hash(0)
private_key = PrivateKey(secret=8675309)
der = private_key.sign(z).der()
sig = der + SIGHASH_ALL.to_bytes(1, 'big')
sec = private_key.point.sec()
script_sig = Script([sig, sec])
transaction.tx_ins[0].script_sig = script_sig 
print(transaction.serialize().hex())

0100000001813f79011acb80925dfe69b3def355fe914bd1d96a3f5f71bf8303c6a989c7d1000000006a47304402207db2402a3311a3b845b038885e3dd889c08126a8570f26a844e3e4049c482a11022010178cdca4129eacbeab7c44648bf5ac1f9cac217cd609d216ec2ebc8d242c0a012103935581e52c354cd2f484fe8ed83af7a3097005b2f9c60bff71d35bd795f54b67feffffff02a135ef01000000001976a914bc3b654dca7e56b04dca18f2566cdaf02e8d9ada88ac99c39800000000001976a9141c4bc762dd5423e332166702cb75f40df79fea1288ac19430600


In [11]:
from ecc import PrivateKey
from helper import hash256, little_endian_to_int
secret = little_endian_to_int(hash256(b'Jimmy Song secret'))
private_key = PrivateKey(secret)
print(private_key.point.address(testnet=True))

mn81594PzKZa9K3Jyy1ushpuEzrnTnxhVg


### Exercise 3

Write the `sign_input` method for the `Tx` class.

#### Make [this test](/edit/code-ch07/tx.py) pass: `tx.py:TxTest:test_sign_input`

In [12]:
# Exercise 3

reload(tx)
run(tx.TxTest("test_sign_input"))

.
----------------------------------------------------------------------
Ran 1 test in 0.062s

OK


### Exercise 4

Create a testnet transaction that sends 60% of a single UTXO to `mwJn1YPMq7y5F8J3LkC5Hxg9PHyZ5K4cFv`. The remaining amount minus fees should go back to your own change address. This should be a 1 input, 2 output transaction.

You can broadcast the transaction at https://testnet.blockchain.info/pushtx

In [13]:
# Exercise 4
from ecc import PrivateKey
from helper import decode_base58, SIGHASH_ALL, hash256, little_endian_to_int
from script import p2pkh_script, Script
from tx import TxFetcher, TxIn, TxOut, Tx

# create 1 TxIn and 2 TxOuts

# 1 of the TxOuts should be back to your address
change_address = 'mhjFP1qnekVoRo9tQQDKm3QaqQTa5BDBkF'

# the other TxOut should be to this address
target_address = 'mwJn1YPMq7y5F8J3LkC5Hxg9PHyZ5K4cFv'

# get the private key from the exercise in Chapter 4
# use the SAME secret phrase you used to generate your address
secret = little_endian_to_int(
    hash256(b'test')
)
private_key = PrivateKey(secret)

assert private_key.point.address(testnet=True) == change_address

# change address should be the address generated from Chapter 4
change_h160 = decode_base58(change_address)
change_script = p2pkh_script(change_h160)

# get the prev_tx and prev_index from the transaction where you got
# some testnet coins
prev_tx_id = '4e43d7f8d3c3556ebd6ea24ac8f68e4b2de18b7750ab59d3b9328ca60414fa9e'

prev_tx_obj = TxFetcher.fetch(prev_tx_id, testnet=True)

prev_index = None
prev_amount = None

for i, tx_out in enumerate(prev_tx_obj.tx_outs):
    if tx_out.script_pubkey.serialize() == change_script.serialize():
        prev_index = i
        prev_amount = tx_out.amount
        break

if prev_index is None:
    raise ValueError('UTXO not found')

print('prev_index:', prev_index)
print('prev_amount:', prev_amount)

# create a transaction input for the previous transaction with
# the default ScriptSig and sequence
prev_tx = bytes.fromhex(prev_tx_id)
tx_in = TxIn(prev_tx, prev_index)

# target amount should be 60% of the output amount
target_amount = round(prev_amount * 0.60)

# set the fee to some reasonable amount
fee = 500

# change amount = amount from the prev tx - target amount - fee
change_amount = prev_amount - target_amount - fee

# create a transaction output for the target amount and address
target_h160 = decode_base58(target_address)
target_script = p2pkh_script(target_h160)

target_output = TxOut(
    amount=target_amount,
    script_pubkey=target_script,
)

# create a transaction output for the change amount and address
change_output = TxOut(
    amount=change_amount,
    script_pubkey=change_script,
)

# create the transaction object
tx_obj = Tx(
    1,
    [tx_in],
    [target_output, change_output],
    0,
    testnet=True,
)

# sign the one input in the transaction object using the private key
assert tx_obj.sign_input(0, private_key)

print('target amount:', target_amount)
print('change amount:', change_amount)
print('fee:', tx_obj.fee())
print('valid:', tx_obj.verify())

# print the transaction's serialization in hex
print(tx_obj.serialize().hex())

# broadcast at http://testnet.blockchain.info/pushtx

prev_index: 0
prev_amount: 130143
target amount: 78086
change amount: 51557
fee: 500


NotImplementedError: 

### Exercise 5

Advanced: get some more testnet coins from a testnet faucet and create a 2 input, 1 output transaction. 1 input should be from the faucet, the other should be from the previous exercise, the output can be your own address.

You can broadcast the transaction at https://testnet.blockchain.info/pushtx

In [14]:
# Exercise 5

from ecc import PrivateKey
from helper import decode_base58, SIGHASH_ALL, hash256, little_endian_to_int
from script import p2pkh_script, Script
from tx import TxFetcher, TxIn, TxOut, Tx

# Create 2 TxIns, 1 from the Exercise 4 and 1 from a testnet faucet
# Creat 1 TxOut to the address above
target_address = 'mwJn1YPMq7y5F8J3LkC5Hxg9PHyZ5K4cFv'

# get the private key from the exercise in Chapter 4
secret = little_endian_to_int(
    hash256(b'test')
)
private_key = PrivateKey(secret)

# get the prev_tx and prev_index from the transaction where you got
# some testnet coins
prev_tx_1 = bytes.fromhex(
    '3884b5c896bef026f8130485eb1e7bf8ff033471691c0a88b9ad99b85c928e3e'
)
prev_index_1 = 0

# create the first transaction input with the default ScriptSig and
# sequence
tx_in_1 = TxIn(prev_tx_1, prev_index_1)

# get the prev_tx and prev_index from the transaction in Exercise 4
prev_tx_2 = bytes.fromhex(
    'bce2cc96c9caec46337d2d514bb7acb5c1afcca8970391f2ad2586a0e3493a4c'
)
prev_index_2 = 1

# create the second transaction input with the default ScriptSig and
# sequence
tx_in_2 = TxIn(prev_tx_2, prev_index_2)

# set the fee to some reasonable amount
fee = 700

# target amount should be the sum of the inputs - fee
input_1_amount = tx_in_1.value(testnet=True)
input_2_amount = tx_in_2.value(testnet=True)

target_amount = input_1_amount + input_2_amount - fee

# create a transaction output for the amount and address
target_h160 = decode_base58(target_address)
target_script = p2pkh_script(target_h160)

tx_out = TxOut(
    target_amount,
    target_script
)

tx_obj = Tx(
    1,
    [tx_in_1, tx_in_2],
    [tx_out],
    0,
    testnet=True
)

# sign the first input using the private key
assert tx_obj.sign_input(0, private_key)

# sign the second input using the private key
assert tx_obj.sign_input(1, private_key)

print('input 1:', input_1_amount)
print('input 2:', input_2_amount)
print('total:', input_1_amount + input_2_amount)
print('output:', target_amount)
print('fee:', tx_obj.fee())
print('valid:', tx_obj.verify())

# print the transaction's serialization in hex
print(tx_obj.serialize().hex())

# broadcast at http://testnet.blockchain.info/pushtx

input 1: 1514602607
input 2: 51557
total: 1514654164
output: 1514653464
fee: 700


NotImplementedError: 